# 14 — Final Figures

Creates publication-ready model comparison and precipitation-map figures.

In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore")

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "configs").exists():
            return candidate
    raise FileNotFoundError(
        "Project root was not found. Run this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
CONFIG_DIR = PROJECT_ROOT / "configs"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

for folder in [INTERIM_DIR, PROCESSED_DIR, OUTPUT_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import rasterio

figure_dir = OUTPUT_DIR / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)

results_path = PROCESSED_DIR / "model_tables" / "validation_results.csv"
results = pd.read_csv(results_path)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(results["model"], results["cv_rmse"])
ax.set_ylabel("Cross-validated RMSE")
ax.set_xlabel("Model")
ax.set_title("Model Performance Comparison")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
fig.savefig(figure_dir / "model_comparison_rmse.png", dpi=300)
plt.show()

In [ ]:
prediction_files = sorted((PROCESSED_DIR / "predictions").glob("*.tif"))
if prediction_files:
    example = prediction_files[0]
    with rasterio.open(example) as src:
        raster = src.read(1)
        bounds = src.bounds

    fig, ax = plt.subplots(figsize=(8, 7))
    image = ax.imshow(
        raster,
        extent=[bounds.left, bounds.right, bounds.bottom, bounds.top],
    )
    ax.set_title(example.stem)
    ax.set_xlabel("Easting / Longitude")
    ax.set_ylabel("Northing / Latitude")
    fig.colorbar(image, ax=ax, label="Rainfall (mm)")
    plt.tight_layout()
    fig.savefig(figure_dir / "example_downscaled_map.png", dpi=300)
    plt.show()
else:
    print("No prediction rasters found. Run Notebook 11 first.")